In [22]:
import matplotlib.pyplot as plt
import numpy as np
import qiskit_aer
import scipy.constants as constants
from qiskit import transpile
from qiskit.circuit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit_aer_encore.simulator import generate_aer_simulator
from qiskit_hamiltonian_simulation.time_independent.direct import (
    MomentumDomainEvolutionQuadratic,
    PositionDomainEvolutionQuadratic,
)
from qiskit_hamiltonian_simulation.time_independent.sample_based import (
    KineticEvolutionSampleBased,
    PotentialEvolutionSampleBased,
)
from qiskit_signals.helper_types import EncodingType
from qiskit_signals.quantum_axis import MomentumAxis, PositionAxis
from qiskit_signals.sample_based_signal import ArbitrarySignalForSampleBasedProtocol

In [23]:
num_qubits = 5
dimension = 2**num_qubits

In [24]:
hbar = constants.hbar
mass = 19.0  # Mass of the particle
omega = 2 * constants.pi * 235.0  # angular frequency

In [25]:
x_0 = np.sqrt(hbar / (mass * omega))
p_0 = np.sqrt(hbar * mass * omega)
E_0 = hbar * omega
t_0 = 1 / omega

In [26]:
delta_x = x_0  # Position step size
delta_t = t_0  # Time step size

In [27]:
max_delta_in_position_domain = 10
max_delta_in_fourier_domain = 10

In [28]:
x_axis = PositionAxis(
    num_qubits=num_qubits, delta_x=delta_x, encoding=EncodingType.UNSIGNED
)
p_axis = MomentumAxis.from_position_axis(x_axis, hbar=hbar)

In [29]:
trotter_steps = 5

In [30]:
x_axis.axis_values

array([0.00000000e+00, 6.13108610e-20, 1.22621722e-19, 1.83932583e-19,
       2.45243444e-19, 3.06554305e-19, 3.67865166e-19, 4.29176027e-19,
       4.90486888e-19, 5.51797749e-19, 6.13108610e-19, 6.74419471e-19,
       7.35730331e-19, 7.97041192e-19, 8.58352053e-19, 9.19662914e-19,
       9.80973775e-19, 1.04228464e-18, 1.10359550e-18, 1.16490636e-18,
       1.22621722e-18, 1.28752808e-18, 1.34883894e-18, 1.41014980e-18,
       1.47146066e-18, 1.53277152e-18, 1.59408238e-18, 1.65539325e-18,
       1.71670411e-18, 1.77801497e-18, 1.83932583e-18, 1.90063669e-18])

In [31]:
p_axis.axis_values

array([ 0.00000000e+00,  3.37729220e-16,  6.75458439e-16,  1.01318766e-15,
        1.35091688e-15,  1.68864610e-15,  2.02637532e-15,  2.36410454e-15,
        2.70183376e-15,  3.03956298e-15,  3.37729220e-15,  3.71502141e-15,
        4.05275063e-15,  4.39047985e-15,  4.72820907e-15,  5.06593829e-15,
       -5.40366751e-15, -5.06593829e-15, -4.72820907e-15, -4.39047985e-15,
       -4.05275063e-15, -3.71502141e-15, -3.37729220e-15, -3.03956298e-15,
       -2.70183376e-15, -2.36410454e-15, -2.02637532e-15, -1.68864610e-15,
       -1.35091688e-15, -1.01318766e-15, -6.75458439e-16, -3.37729220e-16])

In [32]:
def gaussian(axis, mu, sigma):
    """Generate a Gaussian wavefunction on the given axis."""
    x = axis.axis_values
    norm_factor = 1 / (sigma * (2 * constants.pi) ** 0.25)
    return norm_factor * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

In [33]:
def kinetic_signal_function(p):
    """Kinetic energy function."""
    return p**2 / (2.0) / mass


def potential_signal_function(x):
    """Harmonic oscillator potential function."""
    return 0.5 * mass * omega**2 * x**2

In [34]:
potential_signal = ArbitrarySignalForSampleBasedProtocol(
    x_axis, potential_signal_function
)
kinetic_signal = ArbitrarySignalForSampleBasedProtocol(p_axis, kinetic_signal_function)

In [35]:
potential_signal.data

array([0.00000000e+00, 7.78563243e-32, 3.11425297e-31, 7.00706918e-31,
       1.24570119e-30, 1.94640811e-30, 2.80282767e-30, 3.81495989e-30,
       4.98280475e-30, 6.30636227e-30, 7.78563243e-30, 9.42061524e-30,
       1.12113107e-29, 1.31577188e-29, 1.52598396e-29, 1.75176730e-29,
       1.99312190e-29, 2.25004777e-29, 2.52254491e-29, 2.81061331e-29,
       3.11425297e-29, 3.43346390e-29, 3.76824609e-29, 4.11859955e-29,
       4.48452428e-29, 4.86602027e-29, 5.26308752e-29, 5.67572604e-29,
       6.10393582e-29, 6.54771687e-29, 7.00706918e-29, 7.48199276e-29])

In [36]:
kinetic_signal.data

array([0.00000000e+00, 3.00160594e-33, 1.20064238e-32, 2.70144535e-32,
       4.80256950e-32, 7.50401485e-32, 1.08057814e-31, 1.47078691e-31,
       1.92102780e-31, 2.43130081e-31, 3.00160594e-31, 3.63194319e-31,
       4.32231255e-31, 5.07271404e-31, 5.88314764e-31, 6.75361336e-31,
       7.68411121e-31, 6.75361336e-31, 5.88314764e-31, 5.07271404e-31,
       4.32231255e-31, 3.63194319e-31, 3.00160594e-31, 2.43130081e-31,
       1.92102780e-31, 1.47078691e-31, 1.08057814e-31, 7.50401485e-32,
       4.80256950e-32, 2.70144535e-32, 1.20064238e-32, 3.00160594e-33])

In [37]:
delta_ts = np.ones(trotter_steps) * delta_t
len(delta_ts)

5

In [38]:
psi_reg = QuantumRegister(num_qubits, name=r"\psi")
phi_reg = QuantumRegister(num_qubits, name=r"\phi")
success_flag = ClassicalRegister(num_qubits, name="success_flag")
circuit = QuantumCircuit(psi_reg, phi_reg, success_flag)

circuit.save_statevector(label=f"step_{0}")  # type: ignore

for i, delta_t in enumerate(delta_ts):
    potential_evolution = PotentialEvolutionSampleBased(
        V=potential_signal, t=delta_t, hbar=hbar, max_delta=max_delta_in_position_domain
    )
    print(f"{i}th potential num_of_cycles: {potential_evolution.num_of_cycles}")

    circuit.compose(
        potential_evolution,
        circuit.qubits,
        circuit.clbits,
        inplace=True,
    )

    kinetic_evolution = KineticEvolutionSampleBased(
        T=kinetic_signal, t=delta_t, hbar=hbar, max_delta=max_delta_in_fourier_domain
    )
    print(f"{i}th kinetic num_of_cycles: {kinetic_evolution.num_of_cycles}")

    circuit.compose(
        kinetic_evolution,
        circuit.qubits,
        circuit.clbits,
        inplace=True,
    )

    circuit.save_statevector(label=f"step_{i + 1}")  # type: ignore
# circuit.draw("mpl")

0th potential num_of_cycles: 521
0th kinetic num_of_cycles: 6
1th potential num_of_cycles: 521
1th kinetic num_of_cycles: 6
2th potential num_of_cycles: 521
2th kinetic num_of_cycles: 6
3th potential num_of_cycles: 521
3th kinetic num_of_cycles: 6
4th potential num_of_cycles: 521
4th kinetic num_of_cycles: 6


In [39]:
from qiskit.quantum_info import partial_trace


def process_statevector(statevector):
    """Process the statevector to extract position probabilities."""
    n = statevector.num_qubits
    rho = partial_trace(statevector, np.arange(int(n // 2), n).tolist())
    reduced_state = rho.to_statevector()

    return reduced_state

In [40]:
# simulator = generate_aer_simulator(force_gpu=True)
simulator = generate_aer_simulator()
transpiled_circuit = transpile(circuit, simulator)
job = simulator.run(transpiled_circuit, shots=1)
result = job.result()

In [41]:
counts: dict[int, int] = result.get_counts(circuit).int_outcomes()
assert counts[0] == 1
# counts

In [42]:
statevectors = [
    process_statevector(result.data()[f"step_{i}"]) for i in range(len(delta_ts) + 1)
]

In [43]:
def plot_wavefunction(psi, plot_size_scale=1, normalize=True):
    abs_part = np.abs(psi)
    if normalize:
        abs_part = abs_part / np.linalg.norm(abs_part)
    phase_part = np.angle(psi)

    num_of_basis = len(abs_part)
    num_of_qubits = (num_of_basis - 1).bit_length()
    x_axis = np.arange(num_of_basis)

    ket_labels = [
        rf"$|{np.binary_repr(x, width=num_of_qubits)}\rangle$" for x in x_axis
    ]
    # example:
    # KET_LABELS = [r'$|000\rangle$', r'$|001\rangle$', r'$|010\rangle$', r'$|011\rangle$', r'$|100\rangle$', r'$|101\rangle$', r'$|110\rangle$', r'$|111\rangle$']

    PHASE_LABELS = [
        r"$-\pi$",
        r"$-\frac{3\pi}{4}$",
        r"$-\frac{\pi}{2}$",
        r"$-\frac{\pi}{4}$",
        r"$0$",
        r"$\frac{\pi}{4}$",
        r"$\frac{\pi}{2}$",
        r"$\frac{3\pi}{4}$",
        r"$\pi$",
    ]

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(2 * 6.4 * plot_size_scale, 1 * 4.8 * plot_size_scale)
    )
    ax1.stem(abs_part, basefmt="C0")
    ax2.stem(phase_part, basefmt="C0")
    ax2.set_ylim(-np.pi - 0.3, np.pi + 0.3)

    ax1.set_xticks(x_axis, ket_labels, rotation=45)
    ax2.set_xticks(x_axis, ket_labels, rotation=45)

    ax2.set_yticks(
        [
            -np.pi,
            -np.pi * 3 / 4,
            -np.pi / 2,
            -np.pi / 4,
            0,
            np.pi / 4,
            np.pi / 2,
            np.pi * 3 / 4,
            np.pi,
        ]
    )
    ax2.set_yticklabels(PHASE_LABELS)

    ax1.set_xlabel(r"$|x\rangle$")
    ax2.set_xlabel(r"$|x\rangle$")

    ax1.set_ylabel(r"$|\psi(x)|$")
    ax2.set_ylabel(r"$\angle\psi(x)$")

In [ ]:
plot_wavefunction(statevectors[5])

IndexError: list index out of range